In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pathlib import Path

SEED = 42
DECAY_GAMMA = 0.03
EB_KAPPA = 500

# Paths
TX_PATH = "../data/curated/merchant_transactions"
PROB_PATH = "../data/tables/merchant_data/consumer_fraud_probability.csv"
OUTPUT_DIR = "../artifacts/fraud_outputs"
MERCHANTS_PATH = "../data/tables/merchant_data/tbl_merchants.parquet"
CURATED_DIR = "../data/curated"

conf = (
    SparkConf()
    .setAppName("fraud_prob_pipeline")
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .set("spark.sql.shuffle.partitions", "200")
    .set("spark.driver.memory", "6g")
    .set("spark.executor.memory", "6g")
)

spark = (
    SparkSession.builder.master("local[*]").config(conf=conf).getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

### Pipeline overview
- Load transactions and direct fraud probabilities
- Clean transactions
- Tiered probability: direct match (Tier A), user-decay imputation (Tier B), model-based imputation (Tier C)
- Aggregate to merchant-level with empirical Bayes shrinkage and compute FraudScore
- Report merchant rankings


In [ ]:
# Global tunables
LAMBDA = 1.0  # EB/fraud score adjustment strength

### Utils (reusable helpers)

Reusable functions to avoid duplication in feature engineering, ML prep/training, scoring, and aggregation. These keep outputs identical while improving readability.


In [ ]:
from typing import Dict, Tuple
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator


def build_features(df: DataFrame) -> DataFrame:
    """Add transaction, user, and merchant features with consistent fills.

    - log_amount, dow, month
    - user 90d windows: txn_count, sum, avg; days since previous
    - merchant 90d windows: txn_count, sum, avg
    - numeric fill to 0 for stability
    """
    base = (
        df.withColumn("log_amount", F.log1p(F.col("dollar_value")))
          .withColumn("dow", F.dayofweek("order_date"))
          .withColumn("month", F.month("order_date"))
    )

    w_user_90 = (
        Window.partitionBy("user_id")
              .orderBy(F.col("order_date").cast("timestamp").cast("long"))
              .rangeBetween(-90 * 86400, 0)
    )
    base = (
        base.withColumn("user_txn_count_90d", F.count(F.lit(1)).over(w_user_90))
            .withColumn("user_sum_90d", F.sum("dollar_value").over(w_user_90))
            .withColumn("user_avg_amount_90d", F.avg("dollar_value").over(w_user_90))
    )

    w_user_prev = Window.partitionBy("user_id").orderBy("order_date")
    base = (
        base.withColumn("prev_date", F.lag("order_date").over(w_user_prev))
            .withColumn("user_days_since_prev", F.datediff("order_date", "prev_date"))
            .drop("prev_date")
    )

    w_merch_90 = (
        Window.partitionBy("merchant_abn")
              .orderBy(F.col("order_date").cast("timestamp").cast("long"))
              .rangeBetween(-90 * 86400, 0)
    )
    base = (
        base.withColumn("m_txn_count_90d", F.count(F.lit(1)).over(w_merch_90))
            .withColumn("m_sum_90d", F.sum("dollar_value").over(w_merch_90))
            .withColumn("m_avg_amount_90d", F.avg("dollar_value").over(w_merch_90))
    )

    numeric_fill = [
        "log_amount",
        "user_txn_count_90d",
        "user_sum_90d",
        "user_avg_amount_90d",
        "user_days_since_prev",
        "m_txn_count_90d",
        "m_sum_90d",
        "m_avg_amount_90d",
        "take_rate_num",
    ]
    base = base.fillna(0, subset=numeric_fill)

    return base


def build_prep_pipeline(categoricals: Tuple[str, str] = ("rev_band", "biz_tags")) -> Pipeline:
    """Create a consistent prep pipeline: index -> one-hot -> assemble."""
    rev_col, biz_col = categoricals
    indexers = [
        StringIndexer(inputCol=rev_col, outputCol=f"{rev_col}_idx", handleInvalid="keep"),
        StringIndexer(inputCol=biz_col, outputCol=f"{biz_col}_idx", handleInvalid="keep"),
    ]
    encoders = [
        OneHotEncoder(
            inputCols=[f"{rev_col}_idx", f"{biz_col}_idx"],
            outputCols=[f"{rev_col}_oh", f"{biz_col}_oh"],
        )
    ]

    feats = [
        "log_amount",
        "dow",
        "month",
        "user_txn_count_90d",
        "user_sum_90d",
        "user_avg_amount_90d",
        "user_days_since_prev",
        "m_txn_count_90d",
        "m_sum_90d",
        "m_avg_amount_90d",
        "take_rate_num",
    ]

    assembler = VectorAssembler(
        inputCols=feats + [f"{rev_col}_oh", f"{biz_col}_oh"],
        outputCol="features",
        handleInvalid="keep",
    )

    return Pipeline(stages=indexers + encoders + [assembler])


def clip01_df(df: DataFrame, label_col: str, pred_col: str) -> DataFrame:
    """Clip label and prediction columns to [0,1] for stable evaluation."""
    return df.withColumn(
        "lbl",
        F.when(F.col(label_col) < 0, 0.0)
         .when(F.col(label_col) > 1, 1.0)
         .otherwise(F.col(label_col)),
    ).withColumn(
        "prd",
        F.when(F.col(pred_col) < 0, 0.0)
         .when(F.col(pred_col) > 1, 1.0)
         .otherwise(F.col(pred_col)),
    )


def evaluate_regression(df: DataFrame, label_col: str, pred_col: str) -> Dict[str, float]:
    """Return MAE and Brier (MSE) on clipped columns."""
    tmp = clip01_df(df, label_col=label_col, pred_col=pred_col)
    mae_eval = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mae")
    mse_eval = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mse")
    return {"mae": mae_eval.evaluate(tmp), "brier": mse_eval.evaluate(tmp)}


def train_lr_gbt(prepared_df: DataFrame, label_col: str = "p_direct") -> Tuple[Dict[str, float], object, DataFrame]:
    """Train LR and GBT on prepared data; return metrics, best model, and best predictions DF."""
    train_df, valid_df = prepared_df.randomSplit([0.8, 0.2], seed=SEED)

    lr = LinearRegression(labelCol=label_col, featuresCol="features", elasticNetParam=0.5, regParam=0.1)
    gbt = GBTRegressor(labelCol=label_col, featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

    lr_model = lr.fit(train_df)
    gbt_model = gbt.fit(train_df)

    pred_lr = lr_model.transform(valid_df)
    pred_gbt = gbt_model.transform(valid_df)

    m_lr = evaluate_regression(pred_lr, label_col=label_col, pred_col="prediction")
    m_gbt = evaluate_regression(pred_gbt, label_col=label_col, pred_col="prediction")

    metrics = {
        "lr_mae": m_lr["mae"],
        "lr_brier": m_lr["brier"],
        "gbt_mae": m_gbt["mae"],
        "gbt_brier": m_gbt["brier"],
    }

    best_is_gbt = metrics["gbt_mae"] <= metrics["lr_mae"]
    best_model = gbt_model if best_is_gbt else lr_model
    best_valid = pred_gbt if best_is_gbt else pred_lr
    return metrics, best_model, best_valid


def score_unlabeled(base_df: DataFrame, prep_model, model) -> DataFrame:
    """Prepare and score unlabeled rows; add p_model_calibrated clipped to [0,1]."""
    prepared_unlabeled = prep_model.transform(base_df)
    scored = model.transform(prepared_unlabeled)
    return scored.withColumn(
        "p_model_calibrated",
        F.when(F.col("prediction") < 0, 0.0)
         .when(F.col("prediction") > 1, 1.0)
         .otherwise(F.col("prediction")),
    )


def coalesce_probabilities(joined_b: DataFrame, scored_unlabeled: DataFrame) -> Tuple[DataFrame, float]:
    """Combine Tier A/B/C to final p_hat with clipping and global-mean fallback."""
    coalesced = (
        joined_b
        .join(
            scored_unlabeled.select("merchant_abn", "user_id", "order_date", "p_model_calibrated"),
            ["merchant_abn", "user_id", "order_date"],
            "left",
        )
        .withColumn("p_hat", F.coalesce(F.col("p_direct"), F.col("p_user_decay"), F.col("p_model_calibrated")))
    )
    mu = coalesced.select(F.mean("p_hat")).first()[0]
    coalesced = coalesced.withColumn("p_hat", F.when(F.col("p_hat").isNull(), F.lit(mu)).otherwise(F.col("p_hat")))
    coalesced = coalesced.withColumn(
        "p_hat",
        F.when(F.col("p_hat") < 0, 0.0)
         .when(F.col("p_hat") > 1, 1.0)
         .otherwise(F.col("p_hat")),
    )

    per_tx_out = coalesced.select(
        "merchant_abn",
        "user_id",
        F.col("order_date").alias("order_datetime"),
        "dollar_value",
        "biz_tags",
        "rev_band",
        "take_rate",
        "segment",
        "p_hat",
    )
    return per_tx_out, mu


def aggregate_merchants(per_tx_out: DataFrame, merchants_path: str, eb_kappa: float) -> DataFrame:
    """Aggregate KPIs per merchant and apply EB shrinkage."""
    mu = per_tx_out.select(F.avg("p_hat").alias("mu")).first()[0]

    merchants_scope = (
        spark.read.parquet(merchants_path)
             .select("merchant_abn")
             .distinct()
    )

    agg_tx = (
        per_tx_out.groupBy("merchant_abn")
                  .agg(
                      F.count(F.lit(1)).alias("n_txn"),
                      F.sum("dollar_value").alias("sum_amount"),
                      F.avg("p_hat").alias("mean_p"),
                      F.sum(F.col("p_hat") * F.col("dollar_value")).alias("EFL"),
                  )
    )

    agg_complete = (
        merchants_scope.join(agg_tx, "merchant_abn", "left")
            .withColumn("n_txn", F.coalesce(F.col("n_txn"), F.lit(0)))
            .withColumn("sum_amount", F.coalesce(F.col("sum_amount"), F.lit(0.0)))
            .withColumn("EFL", F.coalesce(F.col("EFL"), F.lit(0.0)))
    )

    agg_complete = agg_complete.withColumn(
        "EFLR",
        F.when(F.col("sum_amount") == 0, F.lit(0.0)).otherwise(F.col("EFL") / F.col("sum_amount")),
    )

    agg_complete = agg_complete.withColumn(
        "eb_p",
        F.when(F.col("n_txn") == 0, F.lit(mu)).otherwise(
            (F.col("n_txn") * F.col("mean_p") + F.lit(eb_kappa) * F.lit(mu)) / (F.col("n_txn") + F.lit(eb_kappa))
        ),
    )

    agg_complete = agg_complete.withColumn(
        "se_mean_p",
        F.when(F.col("n_txn") == 0, F.lit(0.0)).otherwise(
            F.sqrt(F.col("mean_p") * (1 - F.col("mean_p")) / F.greatest(F.col("n_txn"), F.lit(1)))
        ),
    )

    return agg_complete


def rank_merchants(agg_complete: DataFrame, merchants_sel: DataFrame) -> Tuple[DataFrame, DataFrame, DataFrame]:
    """Return best_100 and agg_named (full)."""
    agg_named = agg_complete.join(F.broadcast(merchants_sel), "merchant_abn", "left")

    best_100 = (
        agg_named.orderBy(F.col("eb_p").asc())
                 .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
                 .limit(100)
    )

    return best_100, None, agg_named


### Load & Basic Hygiene


In [ ]:
# Read inputs

# Define schemas for strict typing
schema_tx = T.StructType([
    T.StructField("merchant_abn", T.LongType(), True),
    T.StructField("user_id", T.LongType(), True),
    T.StructField("dollar_value", T.DoubleType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("business", T.StringType(), True),
    T.StructField("biz_tags", T.StringType(), True),
    T.StructField("rev_band", T.StringType(), True),
    T.StructField("take_rate", T.StringType(), True),
    T.StructField("segment", T.StringType(), True),
])

schema_prob = T.StructType([
    T.StructField("user_id", T.LongType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("fraud_probability", T.DoubleType(), True),
])

# Load primary inputs
transactions = spark.read.schema(schema_tx).parquet(TX_PATH)
probs = spark.read.option("header", True).schema(schema_prob).csv(PROB_PATH)

# Normalize probabilities to [0,1] if given as percent
probs = probs.withColumn(
    "fraud_probability",
    F.when(F.col("fraud_probability") > 1.0, F.col("fraud_probability") / F.lit(100.0)).otherwise(F.col("fraud_probability"))
)

# Text hygiene & parse merchant take_rate to numeric
transactions = (
    transactions
    .withColumn("biz_tags", F.trim(F.regexp_replace(F.col("biz_tags"), "\s+", " ")))
    .withColumn("rev_band", F.trim(F.col("rev_band")))
    .withColumn("take_rate", F.trim(F.col("take_rate")))
    .withColumn("segment", F.trim(F.col("segment")))
    .withColumn(
        "take_rate_num",
        F.when(F.col("take_rate").rlike(r"^[0-9.]+%$"), F.regexp_replace("take_rate", "%", "").cast("double") / 100.0)
         .when(F.col("take_rate").rlike(r"^[0-9.]+$"), F.col("take_rate").cast("double"))
         .otherwise(F.lit(None).cast("double"))
    )
)

# Cache & counts
transactions.cache(); probs.cache()
print("Transactions:", transactions.count())
print("Probs rows:", probs.count())


In [ ]:
transactions.show(20)
probs.show(20)

### Tier A: Direct probability matches


In [ ]:
# Left join on (user_id, order_datetime) to get p_direct

transactions = transactions.withColumn("order_date", F.col("order_datetime"))
probs = probs.withColumn("order_date", F.col("order_datetime")).drop("order_datetime")

joined_a = transactions.join(
    probs.select("user_id", "order_date", F.col("fraud_probability").alias("p_direct")),
    ["user_id", "order_date"],
    "left",
)

print("Tier A: p_direct non-null:", joined_a.filter(F.col("p_direct").isNotNull()).count())
joined_a.cache()


In [ ]:
# How many distinct user-day keys in probs?
probs_pairs = probs.select("user_id", "order_date").distinct().count()

# How many transaction rows got a direct match?
matches_rows = joined_a.filter(F.col("p_direct").isNotNull()).count()

# How many distinct user-day pairs among those matches?
matches_pairs = (
    joined_a.filter(F.col("p_direct").isNotNull())
            .select("user_id", "order_date")
            .distinct()
            .count()
)

print("probs distinct pairs:", probs_pairs)
print("matched rows:", matches_rows)
print("matched distinct pairs:", matches_pairs)
print("avg tx per matched pair:", matches_rows / matches_pairs)

In [ ]:
from pyspark.sql import functions as F

num_users = transactions.select("user_id").where(
    F.col("user_id").isNotNull()).distinct().count()

num_transactions = transactions.select()
num_merchants = transactions.select("merchant_abn").where(
    F.col("merchant_abn").isNotNull()).distinct().count()

print(f"Distinct users (transactions): {num_users}")
print(f"Distinct merchants (transactions): {num_merchants}")

In [ ]:
from pyspark.sql import functions as F

# All merchants seen in transactions
merchants_all = (
    transactions.select("merchant_abn")
    .where(F.col("merchant_abn").isNotNull())
    .distinct()
)

# Merchants with at least one p_direct
merchants_with_p = (
    joined_a.where(F.col("p_direct").isNotNull())
            .select("merchant_abn")
            .where(F.col("merchant_abn").isNotNull())
            .distinct()
)

# Merchants with no p_direct at all (anti-join)
merchants_no_p = merchants_all.join(
    merchants_with_p, "merchant_abn", "left_anti")

print("Total merchants:", merchants_all.count())
print("Merchants with any p_direct:", merchants_with_p.count())
print("Merchants with no p_direct at all:", merchants_no_p.count())

In [ ]:
from pyspark.sql import functions as F

# Reuse the same merchant sets you derived
merchants_all = (
    transactions.select("merchant_abn")
    .where(F.col("merchant_abn").isNotNull())
    .distinct()
)

merchants_with_p = (
    joined_a.where(F.col("p_direct").isNotNull())
            .select("merchant_abn")
            .where(F.col("merchant_abn").isNotNull())
            .distinct()
)

merchants_no_p = merchants_all.join(
    merchants_with_p, "merchant_abn", "left_anti")

# Transaction counts per group (restrict to non-null merchant_abn for consistency)
tx_nonnull_merch = transactions.where(
    F.col("merchant_abn").isNotNull()).count()
tx_with_p_merchants = transactions.join(F.broadcast(
    merchants_with_p), "merchant_abn", "inner").count()
tx_no_p_merchants = transactions.join(F.broadcast(
    merchants_no_p), "merchant_abn", "inner").count()

print(
    f"Transactions with merchants having any p_direct: {tx_with_p_merchants}")
print(
    f"Transactions with merchants having no p_direct at all: {tx_no_p_merchants}")
print(f"Total transactions with non-null merchant_abn: {tx_nonnull_merch}")
print(
    f"Sanity check (sum matches): {tx_with_p_merchants + tx_no_p_merchants == tx_nonnull_merch}")

# Optional shares
print(f"Share (any p_direct): {tx_with_p_merchants/tx_nonnull_merch:.2%}")
print(f"Share (no p_direct):  {tx_no_p_merchants/tx_nonnull_merch:.2%}")

### Tier B: User-level propensity with time decay


In [ ]:
# Compute p_user_decay for rows without p_direct

# Prepare per-user probability history as (user_id, d_k, p_k)
prob_hist = probs.select(
    "user_id", F.col("order_date").alias("d_k"), F.col("fraud_probability").alias("p_k")
)

# For efficiency: join only for users present in transactions lacking p_direct
users_needing = joined_a.filter(F.col("p_direct").isNull()).select("user_id").distinct()

cand = (
    joined_a.select("user_id", "order_date")
    .join(users_needing, "user_id", "inner")
    .join(prob_hist, "user_id", "inner")
)

# Compute weights w_k = exp(-gamma * |t - d_k|) in days
diff_days = F.abs(F.datediff(F.col("order_date"), F.col("d_k")))
cand = cand.withColumn("w_k", F.exp(-DECAY_GAMMA * diff_days))

# Aggregate per (user_id, order_date)
p_user_decay_df = (
    cand.groupBy("user_id", "order_date")
    .agg(
        (F.sum(F.col("p_k") * F.col("w_k")) / F.sum(F.col("w_k"))).alias("p_user_decay")
    )
)

# Join back onto joined_a, only where p_direct is null
joined_b = (
    joined_a
    .join(p_user_decay_df, ["user_id", "order_date"], "left")
    .withColumn("p_user_decay", F.when(F.col("p_direct").isNull(), F.col("p_user_decay")).otherwise(F.lit(None)))
)

print("Tier B: filled via decay:", joined_b.filter(F.col("p_user_decay").isNotNull()).count())
joined_b.cache()


### Tier C: Model to impute remaining probabilities (soft-label regression)


### Tier C Features


In [ ]:
# Pre-training feature significance checks
# - Pearson correlation vs p_direct for numeric features

# Build minimal feature base if not present
if 'base' not in locals():
    ds = joined_b if 'joined_b' in locals() else joined_a
    base = build_features(ds)

# Use labeled rows (where p_direct is available)
if 'labeled' not in locals():
    labeled = base.filter(F.col("p_direct").isNotNull()).cache()

print("Labeled rows for significance checks:", labeled.count())

numeric_features = [
    "log_amount",
    "dow",
    "month",
    "user_txn_count_90d",
    "user_sum_90d",
    "user_avg_amount_90d",
    "user_days_since_prev",
    "m_txn_count_90d",
    "m_sum_90d",
    "m_avg_amount_90d",
    "take_rate_num",
]

# Pearson correlations
corr_results = []
for feat in numeric_features:
    try:
        c = labeled.stat.corr(feat, "p_direct")
    except Exception:
        c = None
    corr_results.append((feat, c))

# Print correlations sorted by absolute value
corr_results_sorted = sorted(
    [(f, c) for f, c in corr_results if c is not None], key=lambda x: abs(x[1]), reverse=True
)
print("Pearson correlation with p_direct (top):")
for f, c in corr_results_sorted:
    print(f"  {f}: {c:.6f}")


In [ ]:
# Feature engineering for modeling 

# Base for modeling
base = joined_b.withColumn(
    "p_label", F.coalesce(F.col("p_direct"), F.lit(None))
)

# Feature engineering via helper
base = build_features(base)

# Prep pipeline via helper
prep_pipeline = build_prep_pipeline()

# Labeled training data: where p_direct is available
labeled = base.filter(F.col("p_direct").isNotNull()).cache()
print("Labeled rows:", labeled.count())


In [ ]:

# Prepare ablation dataset (no extra income features are used anywhere)
prep_pipeline_no_income = build_prep_pipeline()
prepared_no_income = prep_pipeline_no_income.fit(labeled).transform(labeled)

# Train/evaluate via helpers
metrics_ablate, best_model_ablate, valid_pred_ablate = train_lr_gbt(prepared_no_income, label_col="p_direct")
print("Ablation metrics (no income):", metrics_ablate)

# If ablation is worse than original, try simpler GBT / stronger LR (preserve behavior)
try:
    baseline_metrics = metrics  # from previous training cell
except NameError:
    baseline_metrics = None

if baseline_metrics is None or (metrics_ablate.get("gbt_mae", 1e9) > baseline_metrics.get("gbt_mae", 1e9)):
    print("Ablation worse than baseline; simplifying models...")
    # Reuse prepared_no_income; train tuned models directly
    lr_tuned = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.7, regParam=0.3)
    gbt_tuned = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=4, maxIter=40, stepSize=0.08, subsamplingRate=0.8)

    train_df_no_income, valid_df_no_income = prepared_no_income.randomSplit([0.8, 0.2], seed=SEED)
    lr_model_tuned = lr_tuned.fit(train_df_no_income)
    gbt_model_tuned = gbt_tuned.fit(train_df_no_income)

    pred_lr_tuned = lr_model_tuned.transform(valid_df_no_income)
    pred_gbt_tuned = gbt_model_tuned.transform(valid_df_no_income)

    m_lr_tuned = evaluate_regression(pred_lr_tuned, label_col="p_direct", pred_col="prediction")
    m_gbt_tuned = evaluate_regression(pred_gbt_tuned, label_col="p_direct", pred_col="prediction")

    metrics_tuned = {
        "lr_mae": m_lr_tuned["mae"],
        "lr_brier": m_lr_tuned["brier"],
        "gbt_mae": m_gbt_tuned["mae"],
        "gbt_brier": m_gbt_tuned["brier"],
    }
    print("Tuned metrics (no income):", metrics_tuned)



In [ ]:
# Train LR and GBT; evaluate and calibrate

prepared = prep_pipeline.fit(labeled).transform(labeled)

# Train/evaluate via helper
metrics, best_model, best_valid = train_lr_gbt(prepared, label_col="p_direct")
print(metrics)


In [ ]:
# Apply best model to unlabeled rows and calibrate

prep_model = prep_pipeline.fit(labeled)
prepared = prep_model.transform(labeled)

# Train/evaluate on prepared and pick best model
metrics, best_model, best_valid = train_lr_gbt(prepared, label_col="p_direct")
print("Validation metrics:", metrics)

unlabeled = base.filter(F.col("p_direct").isNull() & F.col("p_user_decay").isNull())

# Score unlabeled with calibrated predictions
scored_unlabeled = score_unlabeled(unlabeled, prep_model, best_model)

print("Scored unlabeled rows:", scored_unlabeled.count())


### Diagnostics


In [ ]:
# Plotting setup
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

PLOTS_DIR = "../plots"
Path(PLOTS_DIR).mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


def save_fig(fig, name: str):
    out = Path(PLOTS_DIR) / f"{name}.png"
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)



In [ ]:
# Calibration curve (reliability diagram)
# Requires best_valid with columns: p_direct (label), prediction (pred)

from pyspark.sql import functions as F

bins = 20
binned = (best_valid
          .withColumn("pred_clipped", F.when(F.col("prediction") < 0, 0.0).when(F.col("prediction") > 1, 1.0).otherwise(F.col("prediction")))
          .withColumn("bin", (F.floor(F.col("pred_clipped") * bins)).cast("int")))

# Guard: put bin==bins into last bin
binned = binned.withColumn("bin", F.when(F.col("bin") >= bins, bins-1).otherwise(F.col("bin")))

cal = (binned.groupBy("bin")
             .agg(F.avg("pred_clipped").alias("mean_pred"), F.avg("p_direct").alias("mean_obs"))
             .orderBy("bin"))

cal_pd = cal.toPandas()

fig, ax = plt.subplots(figsize=(6,4))
ax.plot([0,1],[0,1], linestyle='--', color='gray', label='Ideal')
ax.plot(cal_pd["mean_pred"], cal_pd["mean_obs"], marker='o', label='Model')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Mean observed probability')
ax.set_title('Calibration Curve (validation)')
ax.legend()
save_fig(fig, 'calibration_curve')



In [ ]:
# Lift/decile and error distribution (validation)
from pyspark.sql.window import Window

val_scored = best_valid.select(
    F.col("p_direct").alias("label"),
    F.col("prediction").alias("pred")
).withColumn(
    "pred_clipped", F.when(F.col("pred") < 0, 0.0).when(F.col("pred") > 1, 1.0).otherwise(F.col("pred"))
)

# Deciles by predicted probability
w = Window.orderBy(F.col("pred_clipped").desc())
val_scored = val_scored.withColumn("rank", F.row_number().over(w))
count_val = val_scored.count()
val_scored = val_scored.withColumn("decile", (F.floor((F.col("rank")-1) / (count_val/10))).cast("int"))
val_scored = val_scored.withColumn("decile", F.when(F.col("decile") > 9, 9).otherwise(F.col("decile")))

lift = (val_scored.groupBy("decile")
        .agg(F.avg("label").alias("avg_obs"), F.avg("pred_clipped").alias("avg_pred"))
        .orderBy("decile"))

lift_pd = lift.toPandas()
lift_pd["cum_obs"] = lift_pd.sort_values("decile", ascending=True)["avg_obs"].cumsum()
lift_pd["cum_pred"] = lift_pd.sort_values("decile", ascending=True)["avg_pred"].cumsum()

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(range(1,11), lift_pd.sort_values("decile")["cum_obs"], marker='o', label='Observed (cum)')
ax.plot(range(1,11), lift_pd.sort_values("decile")["cum_pred"], marker='o', label='Predicted (cum)')
ax.set_xlabel('Decile (1=highest risk)')
ax.set_ylabel('Cumulative mean probability')
ax.set_title('Lift by Decile (validation)')
ax.legend()
save_fig(fig, 'lift_decile')

# Error distribution
err_pd = (val_scored.withColumn("abs_err", F.abs(F.col("label") - F.col("pred_clipped"))).select("abs_err").toPandas())
fig, ax = plt.subplots(figsize=(6,4))
sns.histplot(err_pd["abs_err"], bins=50, kde=True, ax=ax)
ax.set_xlabel('|prediction - label|')
ax.set_title('Absolute Error Distribution (validation)')
save_fig(fig, 'error_distribution')



In [ ]:
# Coalesce probabilities and persist per-transaction output

per_tx_out, mu = coalesce_probabilities(joined_b, scored_unlabeled)

print("Per-transaction output rows:", per_tx_out.count())


In [ ]:
# Segment EFLR bar chart (top 10 by EFLR)
from pyspark.sql import functions as F

# Segment EFLR
seg_agg = (
    per_tx_out.where(F.col("segment").isNotNull())
    .groupBy("segment")
    .agg(
        F.sum(F.col("p_hat") * F.col("dollar_value")).alias("EFL"),
        F.sum("dollar_value").alias("sum_amount"),
        F.avg("p_hat").alias("mean_p"),
    )
    .withColumn("EFLR", F.when(F.col("sum_amount") == 0, F.lit(0.0)).otherwise(F.col("EFL")/F.col("sum_amount")))
    .orderBy(F.col("EFLR").desc())
)
seg_pd = seg_agg.limit(10).toPandas()
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=seg_pd, x="EFLR", y="segment", ax=ax)
ax.set_title('Top 10 Segments by EFLR')
ax.set_xlabel('Expected Fraud Loss Rate (EFLR)')
save_fig(fig, 'segment_eflr_bar')

In [ ]:
from pyspark.sql import functions as F

# Tiers on joined_b: T1 if p_direct present; T2 if decay present; else T3
t1 = joined_b.filter(F.col("p_direct").isNotNull()).count()
t2 = joined_b.filter(F.col("p_direct").isNull() &
                     F.col("p_user_decay").isNotNull()).count()
t3 = joined_b.filter(F.col("p_direct").isNull() &
                     F.col("p_user_decay").isNull()).count()

total = joined_b.count()

print(f"Tier 1 (p_direct): {t1} ({t1/total:.2%})")
print(f"Tier 2 (user-decay): {t2} ({t2/total:.2%})")
print(f"Tier 3 (model): {t3} ({t3/total:.2%})")
print("Sanity check:", t1 + t2 + t3 == total)

### Merchant-level aggregation & Empirical-Bayes shrinkage


In [ ]:
# Aggregate to merchant metrics with full scope and EB shrinkage
agg_complete = aggregate_merchants(per_tx_out, MERCHANTS_PATH, EB_KAPPA)

In [ ]:
# Rankings: Best-100 (safest to onboard)

merchants_sel = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn", F.col("name").alias("merchant_name"))
)

best_100, _, agg_named = rank_merchants(agg_complete, merchants_sel)

print("Best-100 (safest to onboard):")
best_100.show(20, truncate=False)

# Save to curated folder
best_100.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/best_100_merchants.csv")

print("Saved best_100 to curated folder")

In [ ]:
# Save all rankings to curated folder
agg_named.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/merchant_fraud_rankings.csv")

In [ ]:
# Compute most fraudulent segment
# Definition: segment with highest EFLR (loss rate). If tie, higher eb_p, then sum_amount desc

from pyspark.sql import functions as F

seg_metrics = (
    per_tx_out
    .where(F.col("segment").isNotNull())
    .groupBy("segment")
    .agg(
        F.sum(F.col("p_hat") * F.col("dollar_value")).alias("EFL"),
        F.sum("dollar_value").alias("sum_amount"),
        F.avg("p_hat").alias("mean_p"),
    )
    .withColumn("EFLR", F.when(F.col("sum_amount") == 0, F.lit(0.0)).otherwise(F.col("EFL")/F.col("sum_amount")))
)

most_fraudulent = (
    seg_metrics
    .orderBy(F.col("EFLR").desc(), F.col("mean_p").desc(), F.col("sum_amount").desc())
    .limit(1)
)

print("Most fraudulent segment:")
most_fraudulent.show(truncate=False)

